# 分块缩放矩阵乘法

本教程介绍在 NVIDIA 和 AMD GPU 上使用 Triton 实现分块缩放矩阵乘法，覆盖 FP4 和 FP8 格式。

支持 OCP 微缩放格式，例如 MXFP4 和 MXFP8，也包括 NVIDIA GPU 上的 NVFP4，以及 AMD GPU 上的 MXFP4。计算通过 NVIDIA 计算能力 10 的第五代 Tensor Core 或 AMD CDNA4 矩阵核心获得硬件加速。

可以通过 format 参数选择格式，通过矩阵维度和迭代步长配置基准测试。下面的空白代码框对应原文中 FP4、FP8 以及启用 CLC 的运行命令。
---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/10-block-scaled-matmul.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

原教程计划在后续更新中支持混合精度的分块缩放矩阵乘法。

## 背景

### NVIDIA GPU 上缩放因子的预重排

支持 PTX 8.7 及以上版本的 CUDA 设备可以使用分块缩放矩阵乘法指令。为使 Tensor Core 矩阵乘加（MMA）内层循环能够低延迟访问缩放因子，需要按照实际访问模式，将缩放因子组织成连续内存布局。

这些指令计算：

$$
C=(A\odot S_A)@ (B\odot S_B)
$$

其中 $S_A$ 和 $S_B$ 分别表示对 A 和 B 广播后的缩放因子。每个缩放因子通常沿相应矩阵的 K 轴广播，与一组元素相乘。每个因子覆盖的元素数量称为向量大小 VEC_SIZE。

在线性行优先布局下，两组缩放因子在全局内存中的形状分别为：

$$
\left(M,\frac{K}{V}\right),
\qquad
\left(N,\frac{K}{V}\right)
\tag{1}
$$

其中 $V=\mathrm{VEC\_SIZE}$；原文形状表达式使用整数除法。

为了避免非连续访问，可以改用打包的分块布局。左侧矩阵的缩放因子布局为：

$$
\left(\frac{M}{32\cdot4},\frac{K}{V\cdot4},32,4,4\right)
\tag{2}
$$

这样，在遍历 K 方向数据块的内层循环中，每次 Tensor Core MMA 对于 A 的 BLOCK_M × BLOCK_K 子块，都可以沿 M 方向连续访问覆盖 128 行的缩放因子块。

为了符合 Triton 中 dot_scaled 的语义，缩放因子先按上述五维布局准备，再通过逻辑转置与重塑，变为该操作期望的二维布局。

缩放因子布局的详细说明：

1. [PTX：tcgen05 MMA 缩放因子 A 的布局](https://docs.nvidia.com/cuda/parallel-thread-execution/#tcgen05-mma-scale-factor-a-layout-1x)。
2. [cuBLAS：一维分块缩放因子的布局](https://docs.nvidia.com/cuda/cublas/#d-block-scaling-factors-layout)。